# Data

In [ ]:
# imports, device
!pip install torchmetrics
import torch
import tokenizers
import transformers
import torchmetrics
from torch import nn
from datasets import load_dataset
from torch.utils.data import DataLoader

if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
device

In [ ]:
# imdb dataset, dataset split
imdb_dataset = load_dataset("imdb")
split = imdb_dataset["train"].train_test_split(train_size=0.8, seed=69)
imdb_train_set, imdb_valid_set = split["train"], split["test"]
imdb_test_set = imdb_dataset["test"]
imdb_train_set.shape

In [ ]:
# tokenizer download
bert_tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")

# in fact, I don't need either of those:
#bert_encoding = bert_tokenizer(train_reviews[:3], padding=True, truncation=True, max_length=500, return_tensors="pt")
# but not first three train_reviews, but something else!
#train_reviews = [review["text"].lower() for review in imdb_train_set] # ?



In [ ]:
# minibatch splitting and tokenizer processing

def collate_fn(batch, tokenizer=bert_tokenizer):
  reviews= [review["text"] for review in batch]
  labels = [[review["label"]] for review in batch]
  encodings = tokenizer(reviews, padding=True, truncation=True, max_length=200, return_tensors="pt")
  # bert_tokenizer instead?, right we already specify it. So in the past, instead of 500, it should be 200,
  # in fact we can remove that thing entirely we were just showing how it works.
  labels = torch.tensor(labels, dtype=torch.float32)
  return encodings, labels

batch_size = 256
imdb_train_loader = DataLoader(imdb_train_set, batch_size=batch_size, collate_fn=collate_fn, shuffle=True)
imdb_valid_loader = DataLoader(imdb_valid_set, batch_size=batch_size, collate_fn=collate_fn)
imdb_test_loader = DataLoader(imdb_valid_set, batch_size=batch_size, collate_fn=collate_fn)


# Model

In [ ]:
# define training and metrics function
# (reusing train and metrics functions from past notebooks: )
def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, patience=2, factor=0.5, epoch_callback=None):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=patience, factor=factor)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        if epoch_callback is not None:
            epoch_callback(model, epoch)
        for index, (X_batch, y_batch) in enumerate(train_loader):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
            train_metric = metric.compute().item()
            print(f"\rBatch {index + 1}/{len(train_loader)}", end="")
            print(f", loss={total_loss/(index+1):.4f}", end="")
            print(f", {train_metric=:.2%}", end="")
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)
        print(f"\rEpoch {epoch + 1}/{n_epochs},                      "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.2%}, "
              f"valid metric: {history['valid_metrics'][-1]:.2%}")
    return history

In [ ]:
# embedding download?

In [ ]:
# define the model
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class SentimentAnalysisModel(nn.Module):
  def __init__(self, vocab_size, n_layers = 2, embed_dim=128, hidden_dim=64, pad_id=0, dropout=0.2):
    super().__init__()
    self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
    self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True, dropout=dropout)
    self.output = nn.Linear(hidden_dim, 1)

  def forward(self, encodings):
    embeddings = self.embed(encodings["input_ids"])

    #_outputs, hidden_states = self.gru(embeddings)
    # ignoring padding tokens with packed sequence data structure instead of a regular tensor
    lengths = encodings["attention_mask"].sum(dim=1)
    packed = pack_padded_sequence(embeddings, lengths=lengths.cpu(), batch_first=True, enforce_sorted=False)
    _outputs, hidden_states = self.gru(packed)

    return self.output(hidden_states[-1])



In [ ]:
# run the training
torch.manual_seed(69)

vocab_size = bert_tokenizer.vocab_size
imdb_model = SentimentAnalysisModel(vocab_size).to(device)

n_epochs = 10
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model.parameters())
accuracy = torchmetrics.Accuracy(task="binary").to(device)

history = train(imdb_model, optimizer, xentropy, accuracy,
                imdb_train_loader, imdb_valid_loader, n_epochs)

In [ ]:
# also here's bidirectional GRU
class SentimentAnalysisModelBidi(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=128,
                 hidden_dim=64, pad_id=0, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim,
                                  padding_idx=pad_id)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout, bidirectional=True)  # <= line changed
        self.output = nn.Linear(2 * hidden_dim, 1)                               # <= line changed

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        lengths = encodings["attention_mask"].sum(dim=1)
        packed = pack_padded_sequence(embeddings, lengths=lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        _outputs, hidden_states = self.gru(packed)
        n_dims = self.output.in_features                                          # <= line added
        top_states = hidden_states[-2:].permute(1, 0, 2).reshape(-1, n_dims)      # <= line added
        return self.output(top_states)                                            # <= line changed
torch.manual_seed(69)

vocab_size = bert_tokenizer.vocab_size
imdb_model_bidi = SentimentAnalysisModelBidi(vocab_size).to(device)

n_epochs = 10
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model_bidi.parameters())
accuracy = torchmetrics.Accuracy(task="binary").to(device)

history = train(imdb_model_bidi, optimizer, xentropy, accuracy, imdb_train_loader,
                imdb_valid_loader, n_epochs)

In [ ]:
# Testing
imdb_model_test_accuracy = evaluate_tm(imdb_model, imdb_test_loader, accuracy).item()
print(f"IMDB Model Test Accuracy: {imdb_model_test_accuracy:.2%}")

imdb_model_bidi_test_accuracy = evaluate_tm(imdb_model_bidi, imdb_test_loader, accuracy).item()
print(f"IMDB Bidirectional Model Test Accuracy: {imdb_model_bidi_test_accuracy:.2%}")

# Deployment

In [ ]:
# clean GPU RAM:
#Out.clear()  # clear Jupyter's `Out` variable which saves all the cell outputs
#del_vars(["albert_token_ids", "attention_mask", "bpe_batch_ids", "encoded_text",
          "lengths", "optimizer", "padded", "probs", "samples", "sequences",
          "x", "xentropy", "y", "Y_logits"])

In [ ]:
# inference
example_reviews = [
    "This movie was absolutely fantastic! The acting was superb, and the story was incredibly engaging. Highly recommend!",
    "What a waste of time. The plot was nonsensical, the characters were flat, and the ending was utterly disappointing. Do not watch.",
    "It was an okay movie. Some parts were good, others not so much. I wouldn't say it's a must-see, but it wasn't terrible either."
]

example_encodings = bert_tokenizer(example_reviews, padding=True, truncation=True, max_length=200, return_tensors="pt")
example_encodings = example_encodings.to(device)

imdb_model.eval()
with torch.no_grad():
    raw_predictions = imdb_model(example_encodings)
    probabilities = torch.sigmoid(raw_predictions)

print("Sentiment Predictions for Example Reviews:")
for i, review in enumerate(example_reviews):
    sentiment_prob = probabilities[i].item()
    print(f"\nReview: {review}")
    print(f"Predicted Sentiment Probability: {sentiment_prob:.2%}")

imdb_model_bidi.eval()
with torch.no_grad():
    raw_predictions_bidi = imdb_model_bidi(example_encodings)
    probabilities_bidi = torch.sigmoid(raw_predictions_bidi)

print("Sentiment Predictions for Example Reviews (Bidirectional Model):")
for i, review in enumerate(example_reviews):
    sentiment_prob_bidi = probabilities_bidi[i].item()
    print(f"\nReview: {review}")
    print(f"Predicted Sentiment Probability (Bidirectional): {sentiment_prob_bidi:.2%}")


In [ ]:
# save the model
from google.colab import drive
drive.mount('/content/drive')

import os

drive_path = '/content/drive/My Drive/Colab_Models'

# Create the directory if it doesn't exist
os.makedirs(drive_path, exist_ok=True)

torch.save(imdb_model_bidi.state_dict(), os.path.join(drive_path, "imdb_model_bidi.pt"))
print(f"imdb_model_bidi saved to {os.path.join(drive_path, 'imdb_model_bidi.pt')}")

torch.save(imdb_model.state_dict(), os.path.join(drive_path, "imdb_model.pt"))
print(f"imdb_model saved to {os.path.join(drive_path, 'imdb_model.pt')}")